In [6]:
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# CONFIGURATION

In [7]:
project_dir = r"C:\cache\Youtube-ETL_Project"
cleaned_dir = os.path.join(project_dir, "data", "cleaned")
latest_cleaned_path = os.path.join(cleaned_dir, "latest_cleaned_path.txt")

if os.path.exists(latest_cleaned_path):
    with open(latest_cleaned_path, "r", encoding="utf-8") as f:
        cleaned_path = f.read().strip()
else:
    cleaned_files = [
        os.path.join(cleaned_dir, file_name)
        for file_name in os.listdir(cleaned_dir)
        if file_name.startswith("cleaned_youtube_") and file_name.endswith(".csv")
    ]
    if not cleaned_files:
        raise FileNotFoundError(f"No cleaned CSV files found in: {cleaned_dir}")
    cleaned_path = max(cleaned_files, key=os.path.getmtime)

table_name = "youtube_videos"
if_exists_mode = "replace"

print(f"Cleaned CSV: {cleaned_path}")
print(f"Load mode: {if_exists_mode} -> table `{table_name}`")

Cleaned CSV: C:\cache\Youtube-ETL_Project\data\cleaned\cleaned_youtube_Jun_2026.csv
Load mode: replace -> table `youtube_videos`


# DATABASE CONNECTION

In [8]:
load_dotenv()

mysql_host = os.getenv("MYSQL_HOST")
mysql_user = os.getenv("MYSQL_USER")
mysql_password = os.getenv("MYSQL_PASSWORD")
mysql_database = os.getenv("MYSQL_DATABASE")
mysql_port = int(os.getenv("MYSQL_PORT", "3306"))

required_vars = {
    "MYSQL_HOST": mysql_host,
    "MYSQL_USER": mysql_user,
    "MYSQL_PASSWORD": mysql_password,
    "MYSQL_DATABASE": mysql_database,
}

missing_vars = [name for name, value in required_vars.items() if not value]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

encoded_password = quote_plus(mysql_password)
server_url = f"mysql+pymysql://{mysql_user}:{encoded_password}@{mysql_host}:{mysql_port}/?charset=utf8mb4"
database_url = f"mysql+pymysql://{mysql_user}:{encoded_password}@{mysql_host}:{mysql_port}/{mysql_database}?charset=utf8mb4"

server_engine = create_engine(server_url)
with server_engine.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{mysql_database}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"))

engine = create_engine(database_url)
print(f"Connected to MySQL database: {mysql_database}")

Connected to MySQL database: yt_db


# READ CLEANED CSV

In [9]:
if not os.path.exists(cleaned_path):
    raise FileNotFoundError(f"Cleaned CSV not found: {cleaned_path}")

df = pd.read_csv(cleaned_path)

if "video_id" not in df.columns:
    raise ValueError("Column `video_id` is required before loading to MySQL")

print(f"Loaded cleaned CSV shape: {df.shape}")
display(df.head())

Loaded cleaned CSV shape: (100, 21)


,video_id,video_title,channel_id,channel_name,subscriber_count,publish_date,video_url,duration,view_count,like_count,...,tags,category,thumbnail_url,keyword,collection_date,duration_seconds,duration_category,day_cleaned,month_cleaned,year_cleaned
0,GzNMN7Ijd8k,How to Earn ₹1 Lakh Per Month ??@PWEarnersPC,UCiGyWN6DEbnj2alu7iapuKQ,Physics Wallah - Alakh Pandey,14300000,2026-06-07T15:19:19Z,https://www.youtube.com/watch?v=GzNMN7Ijd8k,00:54:21,1489776,60814,...,NaN,Education,https://i.ytimg.com/vi/GzNMN7Ijd8k/hqdefault.jpg,data analyst,2026-08-11,3261,Medium Video (30-60 minutes),7,6,2026
1,68FcZUpgC7w,AI Engineer Full Course | 16+ Hours | Beginne...,UCxZC-5v-UccF6FXGI7VXgoQ,The iScale,349000,2026-06-30T12:35:43Z,https://www.youtube.com/watch?v=68FcZUpgC7w,16:04:15,312651,14622,...,AI Engineer Full Course|AI Engineer|AI Course|...,Education,https://i.ytimg.com/vi/68FcZUpgC7w/hqdefault.jpg,data analyst,2026-08-11,57855,Long Video (1 hour or more),30,6,2026
2,X_IQ41y9jbg,Orientation Session || Earners LAPTOP - Video...,UCDkox5NLaLtrYssKumtfzNA,Earners LAPTOP,57100,2026-06-18T13:44:33Z,https://www.youtube.com/watch?v=X_IQ41y9jbg,00:44:45,182943,4658,...,NaN,People & Blogs,https://i.ytimg.com/vi/X_IQ41y9jbg/hqdefault.jpg,data analyst,2026-08-11,2685,Medium Video (30-60 minutes),18,6,2026
3,MOzEvNYvbik,Data Analyst Bootcamp (FREE 35+ Hour Full Cour...,UCLLw7jmFsvfIVaUFsLs8mlQ,Luke Barousse,661000,2026-06-02T12:00:29Z,https://www.youtube.com/watch?v=MOzEvNYvbik,35:37:05,153919,6906,...,NaN,Howto & Style,https://i.ytimg.com/vi/MOzEvNYvbik/hqdefault.jpg,data analyst,2026-08-11,128225,Long Video (1 hour or more),2,6,2026
4,ol9_NnC9-cc,Data Engineer Bootcamp (FREE 27+ Hour Course) ...,UCLLw7jmFsvfIVaUFsLs8mlQ,Luke Barousse,661000,2026-06-15T12:00:33Z,https://www.youtube.com/watch?v=ol9_NnC9-cc,27:19:55,110471,4952,...,NaN,Howto & Style,https://i.ytimg.com/vi/ol9_NnC9-cc/hqdefault.jpg,data analyst,2026-08-11,98395,Long Video (1 hour or more),15,6,2026


# LOAD TO MYSQL

In [10]:
df.to_sql(
    name=table_name,
    con=engine,
    if_exists=if_exists_mode,
    index=False,
    chunksize=1000,
)

with engine.begin() as conn:
    row_count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).scalar_one()

print(f"Loaded {row_count} rows into `{mysql_database}`.`{table_name}`")

Loaded 100 rows into `yt_db`.`youtube_videos`
